In [72]:
%%writefile run.py
"""Condition C2 — Supervised Persona Fine-Tuning runner.

QLoRA fine-tuning of Qwen2.5-7B-Instruct using Unsloth on Kaggle.
"""
from __future__ import annotations
import gc
import os
import sys
from pathlib import Path

# Pin execution
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import mlflow
import yaml
from unsloth import FastLanguageModel, is_bfloat16_supported
from unsloth.chat_templates import get_chat_template
import torch
from datasets import load_dataset
from transformers import TrainingArguments
from trl import SFTTrainer

# Set HF tokens from secrets
from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    hf_username = user_secrets.get_secret("HF_USERNAME")
    os.environ["HF_TOKEN"] = hf_token if hf_token else ""
    os.environ["HF_USERNAME"] = hf_username if hf_username else ""
    print(f"HF Credentials loaded. Username: {os.environ['HF_USERNAME']}", flush=True)
except Exception as e:
    print(f"Warning loading secrets: {e}", flush=True)

sys.path.insert(0, str(Path(__file__).parents[2]))
DEFAULT_CONFIG = Path(__file__).parent / "c2_supervised_finetuning.yaml"

def load_config(path: str | Path) -> dict:
    if os.path.exists(path):
        with open(path) as f:
            return yaml.safe_load(f)
    return {}

def run(config_path: str | Path = DEFAULT_CONFIG) -> None:
    gc.collect()
    torch.cuda.empty_cache()

    if not torch.cuda.is_available():
        print("ERROR: No CUDA GPU detected.", flush=True)
        return
    print(f"Using GPU 0: {torch.cuda.get_device_name(0)} (VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB)", flush=True)

    cfg = load_config(config_path)
    model_cfg = cfg.get("model", {})
    train_cfg = cfg.get("training", {})
    mlflow_cfg = cfg.get("mlflow", {})
    lora_cfg = cfg.get("lora", {})

    mlflow_tracking_uri = os.getenv("MLFLOW_TRACKING_URI", "file:/kaggle/working/mlruns")
    mlflow.set_tracking_uri(mlflow_tracking_uri)
    mlflow.set_experiment(mlflow_cfg.get("experiment_name", "C2_supervised_persona_finetuning"))

    run_name = f"{mlflow_cfg.get('run_name_prefix', 'c2_sft')}_qlora_r{lora_cfg.get('r', 16)}"

    with mlflow.start_run(run_name=run_name):
        mlflow.log_dict(cfg, "config.yaml")

        max_seq_length = train_cfg.get("max_seq_length", 2048)
        base_model_id = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
        print(f"Loading model with FastLanguageModel: {base_model_id}...", flush=True)

        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_id,
            max_seq_length=max_seq_length,
            load_in_4bit=True,
            token=os.environ.get("HF_TOKEN") or None,
        )

        tokenizer = get_chat_template(
            tokenizer,
            chat_template="qwen-2.5",
        )

        # Dataset formatting
        kaggle_data_dir = "/kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/"
        train_dataset_path = os.path.join(kaggle_data_dir, "sft_train.jsonl")
        eval_dataset_path = os.path.join(kaggle_data_dir, "sft_val.jsonl")

        print("Loading and formatting datasets...", flush=True)
        train_dataset = load_dataset("json", data_files=train_dataset_path, split="train")
        eval_dataset = load_dataset("json", data_files=eval_dataset_path, split="train")

        def formatting_prompts_func(examples):
            convos = examples.get("messages", examples.get("conversations", None))
            texts = []
            if convos is not None:
                for convo in convos:
                    text = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
                    texts.append(text)
            elif "text" in examples:
                texts = examples["text"]
            else:
                raise ValueError("Dataset must contain 'messages', 'conversations', or 'text' column.")
            return {"text": texts}

        train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
        eval_dataset = eval_dataset.map(formatting_prompts_func, batched=True)
        print(f"Train samples: {len(train_dataset)} | Eval samples: {len(eval_dataset)}", flush=True)

        print("Setting up LoRA adapter...", flush=True)
        model = FastLanguageModel.get_peft_model(
            model,
            r=lora_cfg.get("r", 16),
            target_modules=lora_cfg.get("target_modules", ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]),
            lora_alpha=lora_cfg.get("lora_alpha", 32),
            lora_dropout=0,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
        )

        output_dir = "/kaggle/working/checkpoints"
        os.makedirs(output_dir, exist_ok=True)
        hub_repo_name = model_cfg.get("hub_repo", "qwen2.5-7b-chioma-sft").split("/")[-1]

        training_args = TrainingArguments(
            output_dir=output_dir,
            num_train_epochs=train_cfg.get("num_train_epochs", 3),
            per_device_train_batch_size=train_cfg.get("per_device_train_batch_size", 1),
            gradient_accumulation_steps=train_cfg.get("gradient_accumulation_steps", 4),
            learning_rate=float(train_cfg.get("learning_rate", 2e-4)),
            lr_scheduler_type=train_cfg.get("lr_scheduler_type", "cosine"),
            warmup_steps=1,
            logging_steps=1,
            save_steps=50,
            save_total_limit=2,
            fp16=not is_bfloat16_supported(),
            bf16=is_bfloat16_supported(),
            report_to="mlflow",
            push_to_hub=False,
        )

        trainer = SFTTrainer(
            model=model,
            train_dataset=train_dataset,
            eval_dataset=eval_dataset,
            dataset_text_field="text",
            tokenizer=tokenizer,
            args=training_args,
            max_seq_length=max_seq_length,
            dataset_num_proc=1,
        )

        print("Starting training loop...", flush=True)
        trainer.train()
        print("Training complete.", flush=True)

        # Save locally
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)

        # Push to Hugging Face
        if os.environ.get("HF_TOKEN") and os.environ.get("HF_USERNAME"):
            full_repo_id = f"{os.environ['HF_USERNAME']}/{hub_repo_name}"
            print(f"Pushing LoRA adapter to Hugging Face: {full_repo_id}...", flush=True)
            model.push_to_hub(full_repo_id, token=os.environ["HF_TOKEN"])
            tokenizer.push_to_hub(full_repo_id, token=os.environ["HF_TOKEN"])
            print(f"\n✅ Adapter uploaded: https://huggingface.co/{full_repo_id}", flush=True)

        if mlflow_cfg.get("log_artifacts", True):
            mlflow.log_artifacts(output_dir, artifact_path="model_checkpoints")

        print("\n✅ SFT run finished successfully.", flush=True)

if __name__ == "__main__":
    run()

Writing run.py


In [73]:
%%writefile c2_supervised_finetuning.yaml
# research/configs/c2_supervised_finetuning.yaml
# Condition C2: Supervised Persona Fine-Tuning — QLoRA on Chioma conversation demos.

experiment:
  name: "C2_supervised_persona_finetuning"
  condition_id: "C2"
  sprint: 4

model:
  base_model_id: "Qwen/Qwen2.5-7B-Instruct"  # HuggingFace model ID
  adapter_type: "qlora"
  # Kaggle specific: Output to /kaggle/working/checkpoints
  output_dir: "/kaggle/working/checkpoints"
  # Kaggle specific: Use user's HF_USERNAME and a unique repo name
  hub_repo: "chioma-sft-v1" # This will become YOUR_HF_USERNAME/chioma-sft-v1

lora:
  r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  target_modules: ["q_proj", "v_proj", "k_proj", "o_proj"]
  bias: "none"
  task_type: "CAUSAL_LM"

quantization:
  load_in_4bit: true
  bnb_4bit_compute_dtype: "bfloat16"
  bnb_4bit_quant_type: "nf4"
  bnb_4bit_use_double_quant: true

training:
  # Kaggle specific: Adjust dataset paths
  dataset: "/kaggle/input/chioma-persona-sft/sft_train.jsonl"
  eval_dataset: "/kaggle/input/chioma-persona-sft/sft_val.jsonl"
  num_train_epochs: 3
  per_device_train_batch_size: 4
  gradient_accumulation_steps: 4           # effective batch = 16
  learning_rate: 2.0e-4
  lr_scheduler_type: "cosine"
  warmup_ratio: 0.05
  max_seq_length: 2048
  fp16: false
  bf16: true
  logging_steps: 10
  eval_steps: 50
  save_steps: 100
  save_total_limit: 3

evaluation:
  metrics:
    - persona_adherence
    - cultural_fluency
    - anti_dependency
    - financial_accuracy
    - urgency
    - rouge_l
    - bert_score_f1
  compare_against: "C1_baseline_prompting"  # baseline experiment to diff against

mlflow:
  experiment_name: "C2_supervised_persona_finetuning"
  run_name_prefix: "c2"
  log_artifacts: true
  log_model: true

Overwriting c2_supervised_finetuning.yaml


In [ ]:
!pip install -q uv
!uv pip install --system \
    mlflow==2.19.0 \
    dvc==3.59.0 \
    dvc-s3==3.2.0 \
    hydra-core==1.3.2 \
    omegaconf==2.3.0 \
    evaluate==0.4.3 \
    bert-score==0.3.13 \
    rouge-score==0.1.2 \
    nltk==3.9.1 \
    scikit-learn==1.6.1 \
    openai==1.59.6 \
    transformers==4.47.1 \
    datasets==2.20.0 \
    pyyaml==6.0.2 \
    matplotlib==3.10.0 \
    seaborn==0.13.2 \
    pytest==8.3.4 \
    python-dotenv==1.0.1 \
    tqdm==4.67.1 \
    peft==0.11.1 \
    bitsandbytes==0.43.1 \
    trl==0.8.6 \
    accelerate==0.32.1 \
    sentencepiece

Using Python 3.12.13 environment at: /usr
Resolved 209 packages in 275ms                                       
Prepared 1 package in 724ms                                              
Uninstalled 3 packages in 12ms
Installed 3 packages in 13ms                                
 - matplotlib==3.11.1
 + matplotlib==3.10.0
 - pyyaml==6.0.3
 + pyyaml==6.0.2
 - tqdm==4.70.0
 + tqdm==4.67.1


In [10]:
!pip install -q uv
!uv pip install --system -U \
    pyOpenSSL \
    cryptography \
    mlflow==2.19.0 \
    dvc==3.59.0 \
    dvc-s3==3.2.0 \
    hydra-core==1.3.2 \
    omegaconf==2.3.0 \
    evaluate==0.4.3 \
    bert-score==0.3.13 \
    rouge-score==0.1.2 \
    nltk==3.9.1 \
    scikit-learn==1.6.1 \
    openai==1.59.6 \
    transformers==4.47.1 \
    datasets==2.20.0 \
    peft==0.11.1 \
    bitsandbytes==0.43.1 \
    trl==0.8.6 \
    accelerate==0.32.1 \
    python-dotenv==1.0.1 \
    sentencepiece

Using Python 3.12.13 environment at: /usr
Resolved 206 packages in 762ms                                       
Checked 206 packages in 3ms


In [14]:
!pip uninstall -y torchvision

Found existing installation: torchvision 0.25.0+cu128
Uninstalling torchvision-0.25.0+cu128:
  Successfully uninstalled torchvision-0.25.0+cu128


In [35]:
# Kaggle Cell 4: Execute the fine-tuning
!python run.py

Loading train dataset from: /kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/sft_train.jsonl
Loading eval dataset from: /kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/sft_val.jsonl


In [ ]:
!pip install -q -U --force-reinstall bitsandbytes

In [33]:
import torch
import bitsandbytes as bnb

print(f"CUDA Available: {torch.cuda.is_available()}")
print(f"GPU Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

# Verify 4-bit CUDA layer functionality
test_layer = bnb.nn.Linear4bit(32, 32, bias=False).cuda()
print("✅ bitsandbytes CUDA binary is loaded and operational.")

CUDA Available: True
GPU Device: Tesla T4
✅ bitsandbytes CUDA binary is loaded and operational.


In [40]:
!python -u run.py

Loading train dataset from: /kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/sft_train.jsonl
Loading eval dataset from: /kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/sft_val.jsonl


In [41]:
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
print("Token present:", bool(secrets.get_secret("HF_TOKEN")))
print("Username present:", bool(secrets.get_secret("HF_USERNAME")))

Token present: True
Username present: True


In [58]:
!python -u run.py

HF Credentials loaded. Username: Danleon56
Using GPU 0: Tesla T4 (VRAM: 15.64 GB)
Loading datasets...
Train samples: 8 | Eval samples: 1
Loading pre-quantized 4-bit model: unsloth/Qwen2.5-7B-Instruct-bnb-4bit...
tokenizer_config.json: 7.36kB [00:00, 21.5MB/s]
vocab.json: 2.78MB [00:00, 14.3MB/s]
merges.txt: 1.67MB [00:00, 10.2MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:01<00:00, 7.75MB/s]
special_tokens_map.json: 100%|█████████████████| 614/614 [00:00<00:00, 3.40MB/s]
config.json: 1.34kB [00:00, 5.65MB/s]
Unused kwargs: ['_load_in_4bit', '_load_in_8bit', 'quant_method']. These kwargs are not used in <class 'transformers.utils.quantization_config.BitsAndBytesConfig'>.


In [69]:
!pip install -q -U unsloth

In [74]:
!python -u run.py

/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:98: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Detected that PyTorch and TorchAudio were compiled with different CUDA versions. PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8. Please install the TorchAudio version that matches your PyTorch version.
  disable_torchaudio_if_cuda_mismatched()
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
HF Credentials loaded. Username: Danleon56
Using GPU 0: Tesla T4 (VRAM: 15.64 GB)
Loading model with FastLanguageModel: unsloth/Qwen2.5-7B-Instruct-bnb-4bit...
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max mem

In [ ]:
import bitsandbytes as bnb
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device count: {torch.cuda.device_count()}")
print(f"Device name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")
# Test 4-bit Linear layer allocation
linear = bnb.nn.Linear4bit(16, 16, bias=False).cuda()
print("✅ bitsandbytes CUDA binary loaded successfully!")

In [71]:
import os

# Remove old files if they exist
for f in ["run.py", "c2_supervised_finetuning.py"]:
    if os.path.exists(f):
        os.remove(f)

In [102]:
%%writefile metrics.py
from __future__ import annotations

import json
import os
import re
import time
from dataclasses import dataclass, field
import mlflow
from openai import OpenAI

_judge_client: OpenAI | None = None

def _get_judge() -> OpenAI:
    global _judge_client
    if _judge_client is None:
        api_key = os.getenv("LLM_API_KEY") or os.getenv("GROQ_API_KEY", "")
        base_url = os.getenv("LLM_BASE_URL", "https://api.groq.com/openai/v1")
        _judge_client = OpenAI(api_key=api_key, base_url=base_url)
    return _judge_client

# Configured for openai/gpt-oss-120b
_JUDGE_MODEL = os.getenv("EVAL_JUDGE_MODEL", "openai/gpt-oss-120b")

@dataclass
class EvalResult:
    persona_adherence: float = 0.0
    cultural_fluency: float = 0.0
    anti_dependency: float = 0.0
    financial_accuracy: float = 0.0
    urgency: float = 0.0
    rouge_l: float = 0.0
    bert_score_f1: float = 0.0
    metadata: dict = field(default_factory=dict)

    def to_dict(self) -> dict:
        return {
            "persona_adherence": self.persona_adherence,
            "cultural_fluency": self.cultural_fluency,
            "anti_dependency": self.anti_dependency,
            "financial_accuracy": self.financial_accuracy,
            "urgency": self.urgency,
            "rouge_l": self.rouge_l,
            "bert_score_f1": self.bert_score_f1,
            **self.metadata,
        }

    @property
    def composite_score(self) -> float:
        return (
            self.persona_adherence * 0.25
            + self.cultural_fluency * 0.20
            + self.anti_dependency * 0.20
            + self.financial_accuracy * 0.20
            + self.urgency * 0.15
        )

_COMBINED_EVAL_PROMPT = """You are an expert evaluator assessing an AI financial mentor response (Persona: Chioma) for African founders and individuals.
Evaluate the AI response across these 5 dimensions on a scale from 0.0 (completely fails) to 1.0 (fully meets):

1. persona_adherence: Warm but direct, gives specific actionable advice with numbers, acknowledges before advising, ends with a single sharp question/next action rather than generic advice.
2. cultural_fluency: Genuine understanding of African financial realities (informal markets, mobile money, SACCOs/chamas/esusu, local context).
3. anti_dependency: Teaches underlying frameworks and principles rather than merely handing over generic lists.
4. financial_accuracy: Factually realistic and sensible numbers/guidance for African markets.
5. urgency: Surfaces the concrete cost of inaction or realistic timelines.

User query: {user_message}
AI output: {response}

Output format: Return ONLY a JSON object with float scores (between 0.0 and 1.0) for:
"persona_adherence", "cultural_fluency", "anti_dependency", "financial_accuracy", "urgency"."""

def score_all_dimensions(user_message: str, response: str) -> dict[str, float]:
    prompt = _COMBINED_EVAL_PROMPT.format(
        user_message=user_message,
        response=response,
    )
    
    defaults = {
        "persona_adherence": 0.0,
        "cultural_fluency": 0.0,
        "anti_dependency": 0.0,
        "financial_accuracy": 0.0,
        "urgency": 0.0,
    }

    client = _get_judge()
    if not client.api_key:
        print("[ERROR] LLM API key missing for judge client.", flush=True)
        return defaults

    for attempt in range(3):
        try:
            res = client.chat.completions.create(
                model=_JUDGE_MODEL,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                max_tokens=512,
                temperature=0.0,
            )

            msg = res.choices[0].message
            raw_text = (msg.content or getattr(msg, "reasoning_content", "") or "").strip()

            # Robust JSON extraction from raw content
            json_match = re.search(r"\{[^{}]*\}", raw_text, re.DOTALL)
            if json_match:
                parsed = json.loads(json_match.group(0))
                return {
                    k: min(max(float(parsed.get(k, 0.0)), 0.0), 1.0)
                    for k in defaults.keys()
                }

            print(f"[Judge Warning] Could not find JSON block in output: {repr(raw_text)}", flush=True)
            break
        except Exception as e:
            time.sleep(1.5 * (attempt + 1))
            if attempt == 2:
                print(f"[Judge API Error with {_JUDGE_MODEL}]: {e}", flush=True)

    return defaults

def score_rouge_l(response: str, reference: str) -> float:
    try:
        from rouge_score import rouge_scorer
        scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
        return scorer.score(reference, response)["rougeL"].fmeasure
    except ImportError:
        return 0.0

def score_bert(response: str, reference: str) -> float:
    try:
        from bert_score import score as bert_score
        _, _, f1 = bert_score([response], [reference], lang="en", verbose=False)
        return float(f1[0])
    except ImportError:
        return 0.0

def evaluate_response(
    user_message: str,
    response: str,
    reference: str | None = None,
) -> EvalResult:
    scores = score_all_dimensions(user_message, response)

    result = EvalResult(
        persona_adherence=scores["persona_adherence"],
        cultural_fluency=scores["cultural_fluency"],
        anti_dependency=scores["anti_dependency"],
        financial_accuracy=scores["financial_accuracy"],
        urgency=scores["urgency"],
    )

    if reference:
        result.rouge_l = score_rouge_l(response, reference)
        result.bert_score_f1 = score_bert(response, reference)
        
    return result

def log_eval_to_mlflow(result: EvalResult, step: int | None = None) -> None:
    mlflow.log_metrics(result.to_dict(), step=step)
    mlflow.log_metric("composite_score", result.composite_score, step=step)

Writing metrics.py


In [103]:
%%writefile evaluate_runner.py
from __future__ import annotations
import gc
import json
import os
import sys
import numpy as np
import pandas as pd
import torch
import mlflow

from kaggle_secrets import UserSecretsClient
try:
    user_secrets = UserSecretsClient()
    # Pull LLM Judge key from secrets if available
    llm_key = user_secrets.get_secret("LLM_API_KEY") or user_secrets.get_secret("GROQ_API_KEY")
    if llm_key:
        os.environ["LLM_API_KEY"] = llm_key
except Exception as e:
    print(f"Notice regarding API secrets: {e}", flush=True)

from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from metrics import evaluate_response, EvalResult

def load_eval_data(file_path: str):
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

def main():
    gc.collect()
    torch.cuda.empty_cache()

    adapter_path = "/kaggle/working/checkpoints"
    data_path = "/kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/sft_val.jsonl"
    
    # Check for test split if available, otherwise val
    test_path = "/kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/sft_test.jsonl"
    if os.path.exists(test_path):
        data_path = test_path

    print(f"Loading evaluation dataset: {data_path}", flush=True)
    samples = load_eval_data(data_path)
    print(f"Total evaluation samples: {len(samples)}", flush=True)

    print("Loading fine-tuned model for inference...", flush=True)
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=adapter_path,
        max_seq_length=2048,
        load_in_4bit=True,
    )
    FastLanguageModel.for_inference(model)

    tokenizer = get_chat_template(
        tokenizer,
        chat_template="qwen-2.5",
    )

    eval_results = []
    
    print("\nStarting generation and scoring...", flush=True)
    for i, sample in enumerate(samples):
        messages = sample.get("messages", sample.get("conversations", []))
        if not messages:
            continue

        # Extract user input and ground truth reference
        user_msg = ""
        ref_msg = ""
        prompt_messages = []
        
        for msg in messages:
            if msg["role"] == "user":
                user_msg = msg["content"]
                prompt_messages.append(msg)
            elif msg["role"] == "assistant":
                ref_msg = msg["content"]

        # Format prompt with chat template for inference
        input_ids = tokenizer.apply_chat_template(
            prompt_messages,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt"
        ).to("cuda")

        with torch.no_grad():
            output = model.generate(
                input_ids=input_ids,
                max_new_tokens=512,
                temperature=0.7,
                top_p=0.9,
                use_cache=True,
            )

        # Slice generated response
        generated_tokens = output[0][input_ids.shape[1]:]
        generated_text = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

        print(f"\n--- Sample {i+1} ---")
        print(f"User: {user_msg}")
        print(f"Model: {generated_text}")
        print(f"Reference: {ref_msg}\n")

        # Evaluate against the rubric & reference
        score: EvalResult = evaluate_response(
            user_message=user_msg,
            response=generated_text,
            reference=ref_msg if ref_msg else None,
        )
        
        eval_results.append(score.to_dict())

    # Aggregate metric averages
    df = pd.DataFrame(eval_results)
    avg_scores = df.mean(numeric_only=True).to_dict()

    # Calculate composite score from averages
    composite = (
        avg_scores.get("persona_adherence", 0.0) * 0.25
        + avg_scores.get("cultural_fluency", 0.0) * 0.20
        + avg_scores.get("anti_dependency", 0.0) * 0.20
        + avg_scores.get("financial_accuracy", 0.0) * 0.20
        + avg_scores.get("urgency", 0.0) * 0.15
    )
    avg_scores["composite_score"] = composite

    # Log to MLflow
    mlflow.set_tracking_uri(os.getenv("MLFLOW_TRACKING_URI", "file:/kaggle/working/mlruns"))
    mlflow.set_experiment("C2_supervised_persona_finetuning")
    with mlflow.start_run(run_name="evaluation_summary"):
        mlflow.log_metrics(avg_scores)

    print("\n================ EVALUATION SUMMARY ================")
    for metric, value in avg_scores.items():
        print(f"{metric:<25}: {value:.4f}")
    print("====================================================\n")

if __name__ == "__main__":
    main()

Overwriting evaluate_runner.py


In [104]:
!python -B -u evaluate_runner.py

/usr/local/lib/python3.12/dist-packages/unsloth/_gpu_init.py:98: UserWarning: Unsloth: torchaudio cannot initialise against this torch and has been disabled for this process, so anything that needs it will report it as missing rather than crash at import. Install the matching wheel to restore it. Original error: Detected that PyTorch and TorchAudio were compiled with different CUDA versions. PyTorch has CUDA version 13.0 whereas TorchAudio has CUDA version 12.8. Please install the TorchAudio version that matches your PyTorch version.
  disable_torchaudio_if_cuda_mismatched()
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading evaluation dataset: /kaggle/input/datasets/danielkusiboateng/chioma-persona-sft/sft_test.jsonl
Total evaluation samples: 1
Loading fine-tuned model for inference...
==((====))==  Unsloth 2026.8.18: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs

In [101]:
import os
import shutil

# Remove compiled pycache folders
for root, dirs, files in os.walk("/kaggle/working"):
    if "__pycache__" in dirs:
        shutil.rmtree(os.path.join(root, "__pycache__"), ignore_errors=True)

# Remove old metrics file to ensure a clean overwrite
if os.path.exists("metrics.py"):
    os.remove("metrics.py")

print("Cleaned cache and removed old metrics.py")

Cleaned cache and removed old metrics.py
